In [ ]:
#NVIDIA TensorRT ve ULtralytics kütüphanelerini yükleyelim.
# NVIDIA TensorRT ve Ultralytics kütüphanelerini yükleyelim
%pip install -q ultralytics onnx onnxscript
%pip install -q tensorrt

import torch
print("GPU Aktif mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Kullanılan GPU:", torch.cuda.get_device_name(0))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.4 MB/s eta 0:00:00
GPU Aktif mi?: True
Kullanılan GPU: Tesla T4


In [ ]:
import time
import torch
import numpy as np
from ultralytics import YOLO

# Model yolunu Colab'daki konuma göre ayarla
MODEL_PATH = "yolov8_gold_best.pt"

DUMMY_INPUT = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

print("1. Model Dönüştürme (Export) Başlatılıyor")
model = YOLO(MODEL_PATH)

# A. ONNX Export
print("\n[Info] PyTorch (.pt) -> ONNX formatına dönüştürülüyor...")
model.export(format="onnx", imgsz=640, dynamic=False)
onnx_path = MODEL_PATH.replace(".pt", ".onnx")

# B. TensorRT Export (Tesla T4 GPU üzerinde FP16 Hassasiyeti ile)
print("\n[INFO] PyTorch (.pt) -> TensorRT (.engine) FP16 formatına dönüştürülüyor...")
# half=True: T4 üzerindeki Tensor Çekirdeklerini aktif eder, hızı katlar!
model.export(format="engine", imgsz=640, device=0, half=True)
engine_path = MODEL_PATH.replace(".pt", ".engine")

print("\n 2. Benchmark Testi (T4 GPU Üzerinde 100 Kare) ")

def run_benchmark(model_file, model_name, iterations=100):
    m = YOLO(model_file, task="detect")

    # Isınma Turu (GPU'yu uyandırmak için)
    for _ in range(10):
        m(DUMMY_INPUT, verbose=False)

    start_time = time.perf_counter()
    for _ in range(iterations):
        m(DUMMY_INPUT, verbose=False)
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_latency_ms = (total_time / iterations) * 1000
    fps = iterations / total_time
    return avg_latency_ms, fps

# Testleri Çalıştır
pt_lat, pt_fps = run_benchmark(MODEL_PATH, "PyTorch (.pt)")
onnx_lat, onnx_fps = run_benchmark(onnx_path, "ONNX Runtime (.onnx)")
trt_lat, trt_fps = run_benchmark(engine_path, "TensorRT (.engine)")

print("\n=======================================================")
print("=== Benchmark Tablosu (Tesla T4) ===")
print("=======================================================")
print(f"PyTorch (.pt)        -> Gecikme: {pt_lat:.2f} ms | FPS: {pt_fps:.1f}")
print(f"ONNX Runtime (.onnx) -> Gecikme: {onnx_lat:.2f} ms | FPS: {onnx_fps:.1f} (Hız: x{pt_lat/onnx_lat:.1f})")
print(f"TensorRT (.engine)   -> Gecikme: {trt_lat:.2f} ms | FPS: {trt_fps:.1f} (Hız: x{pt_lat/trt_lat:.1f})")
print("=======================================================")

1. Model Dönüştürme (Export) Başlatılıyor

[Info] PyTorch (.pt) -> ONNX formatına dönüştürülüyor...
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'yolov8_gold_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 317ms
Prepared 3 packages in 394ms
Installed 3 packages in 10ms
 + colorama==0.4.6
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1

2026-08-09 21:24:55,331 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-08-09 21:24:56,025 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


...

2026-08-09 21:24:56,866 - [modelopt][onnx] - WARNING - Some initializers contain values smaller than smallest fp16 value, values will be replaced with 6.0e-08.


2026-08-09 21:24:57,604 - [modelopt][onnx] - INFO - Converted 229/231 nodes (99.13%) to fp16


INFO:modelopt.onnx.autocast:Converted 229/231 nodes (99.13%) to fp16


TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 5, 8400) DataType.FLOAT
TensorRT: building FP16 engine as yolov8_gold_best.engine
TensorRT: export success ✅ 200.6s, saved as 'yolov8_gold_best.engine' (46.6 MB)

Export complete (203.5s)
Results saved to /content/yolov8_gold_best.engine
Predict:         yolo predict task=detect model=yolov8_gold_best.engine imgsz=640 quantize=16
Validate:        yolo val task=detect model=yolov8_gold_best.engine imgsz=640 data=/content/thermal_gold.yaml quantize=16 
Visualize:       https://netron.app

 2. Benchmark Testi (T4 GPU Üzerinde 100 Kare) 


TypeError: Config() got an unexpected keyword argument 'deprecated'

In [ ]:
!pip install -q onnxruntime-gpu

In [ ]:
import time
import torch
import numpy as np
import onnxruntime as ort
from ultralytics.nn.autobackend import AutoBackend

print("Saf GPU Benchmark (Tesla T4) Başlatılıyor\n")

# 1. ONNX Runtime GPU Benchmark
def benchmark_onnx(onnx_path, iterations=100):
    # CUDA Execution Provider ile ONNX oturumu
    session = ort.InferenceSession(onnx_path, providers=['CUDAExecutionProvider'])
    input_name = session.get_inputs()[0].name

    # 640x640x3 FP32 Matris
    dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)

    # Warmup
    for _ in range(15):
        session.run(None, {input_name: dummy_input})

    start_time = time.perf_counter()
    for _ in range(iterations):
        session.run(None, {input_name: dummy_input})
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_latency = (total_time / iterations) * 1000
    fps = iterations / total_time
    return avg_latency, fps

# 2. TensorRT Enginee Benchmark (FP16 CUDA Stream)
def benchmark_tensorrt(engine_path, iterations=100):
    # AutoBackend saf model yükleyicisidir, torch._dynamo çağırmaz
    model = AutoBackend(engine_path, device=torch.device('cuda:0'))

    # TensorRT FP16 beklediği için half() tensor
    dummy_tensor = torch.zeros((1, 3, 640, 640), device='cuda:0', dtype=torch.float16)

    # Warmup
    for _ in range(15):
        _ = model(dummy_tensor)

    torch.cuda.synchronize()
    start_time = time.perf_counter()
    for _ in range(iterations):
        _ = model(dummy_tensor)
    torch.cuda.synchronize()
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_latency = (total_time / iterations) * 1000
    fps = iterations / total_time
    return avg_latency, fps

# Testleri Çalıştır
try:
    onnx_lat, onnx_fps = benchmark_onnx("yolov8_gold_best.onnx")
    print(f"[BAŞARILI] ONNX Runtime (CUDA) -> Gecikme: {onnx_lat:.2f} ms | FPS: {onnx_fps:.1f}")
except Exception as e:
    print(f"[HATA] ONNX Benchmark: {e}")

try:
    trt_lat, trt_fps = benchmark_tensorrt("yolov8_gold_best.engine")
    print(f"[BAŞARILI] TensorRT (.engine FP16) -> Gecikme: {trt_lat:.2f} ms | FPS: {trt_fps:.1f}")
except Exception as e:
    print(f"[HATA] TensorRT Benchmark: {e}")

=== SAF GPU BENCHMARK (Tesla T4) BAŞLATILIYOR ===



/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  set_provider_options(name, options)


[BAŞARILI] ONNX Runtime (CUDA) -> Gecikme: 110.21 ms | FPS: 9.1
Loading yolov8_gold_best.engine for TensorRT inference...
[BAŞARILI] TensorRT (.engine FP16) -> Gecikme: 2.60 ms | FPS: 384.2
